In [ ]:
"""Homework 4: 3D Letter Shading and Camera — STARTUP Template.

Provided: GLSL vertex/fragment shaders, shader compilation, OpenGL window,
vertex colors, front-face triangles, GPU buffers, perspective projection,
and the mouse callback wiring.

TODOs: back-face triangles, side walls, depth buffering,
mouse-driven yaw/pitch, and model-view matrix composition.

Run locally (not Google Colab):
    pip install numpy glfw PyOpenGL
    python shade_3d_letter_starter.py

The initial program intentionally displays only the front face.
Do not modify the provided shader source.
As in HW3, define vertices and edges for another letter. 
Letter "B" is just an example. 
"""

import sys
import ctypes
import numpy as np
import glfw
from OpenGL.GL import *

# ==============================================================================
# GLSL Shaders (Faceted Shading via dFdx / dFdy)
# ==============================================================================

VERTEX_SHADER_SRC = """#version 330 core
layout(location = 0) in vec3 aPos;
layout(location = 1) in vec3 aColor;

uniform mat4 uModelView;
uniform mat4 uPersp;

out vec3 vPoint;   // Position in eye/camera space
out vec3 vColor;

void main() {
    vPoint = (uModelView * vec4(aPos, 1.0)).xyz;
    gl_Position = uPersp * vec4(vPoint, 1.0);
    vColor = aColor;
}
"""

FRAGMENT_SHADER_SRC = """#version 330 core
in vec3 vPoint;
in vec3 vColor;

out vec4 FragColor;

uniform vec3 uLight = vec3(2.0, 2.0, 3.0);  // Light position in eye space
uniform float uAmb = 0.15;                  // Ambient reflection weight
uniform float uDif = 0.75;                  // Diffuse reflection weight
uniform float uSpc = 0.60;                  // Specular reflection weight

void main() {
    // 1. Analytical face normal via screen-space derivatives (faceted appearance)
    vec3 dx = dFdx(vPoint);
    vec3 dy = dFdy(vPoint);
    vec3 N = normalize(cross(dx, dy));

    // 2. Diffuse term (Lambertian)
    vec3 L = normalize(uLight - vPoint);
    float d = max(0.0, dot(N, L));

    // 3. Specular highlight (Phong reflection)
    vec3 E = normalize(-vPoint);             // Eye vector toward origin in eye space
    vec3 R = reflect(-L, N);
    float h = max(0.0, dot(R, E));
    float s = pow(h, 64.0);

    // 4. Combined illumination
    float intensity = min(1.0, uAmb + uDif * d) + uSpc * s;
    FragColor = vec4(intensity * vColor, 1.0);
}
"""

# ==============================================================================
# Geometry Builder: 3D Extruded Letter 'B'
# ==============================================================================

def build_extruded_letter_b():
    """
    Builds the vertices, colors, and triangle indices for the extruded letter 'B'.
    Outer boundary loop in counter-clockwise order:
      0: J  (0, 0)      - Interior hub vertex
      1: E  (50, -150)  - Bottom-right inner
      2: F  (80, -130)  - Bottom-right corner
      3: G  (80, -50)   - Lower lobe cusp
      4: I  (50, 0)     - Middle notch
      5: H  (80, 50)    - Upper lobe cusp
      6: D  (80, 130)   - Top-right corner
      7: C  (50, 150)   - Top-right inner
      8: A  (-50, 150)  - Top-left
      9: B  (-50, -150) - Bottom-left
    """
    front_2d = np.array([
        [  0.0,    0.0],  # 0: J
        [ 50.0, -150.0],  # 1: E
        [ 80.0, -130.0],  # 2: F
        [ 80.0,  -50.0],  # 3: G
        [ 50.0,    0.0],  # 4: I
        [ 80.0,   50.0],  # 5: H
        [ 80.0,  130.0],  # 6: D
        [ 50.0,  150.0],  # 7: C
        [-80.0,  150.0],  # 8: A
        [-80.0, -150.0]   # 9: B
    ], dtype=np.float32)

    # Provided: center and uniformly scale the 2D letter
    center_xy = (front_2d.max(axis=0) + front_2d.min(axis=0)) / 2.0
    front_2d -= center_xy
    scale = 1.6 / 300.0  # Height is 300
    front_2d *= scale

    z_front = 0.10
    z_back  = -0.10

    # 1. Vertices (Front: 0..9, Back: 10..19)
    front_verts = np.column_stack([front_2d, np.full(10, z_front, dtype=np.float32)])
    back_verts  = np.column_stack([front_2d, np.full(10, z_back,  dtype=np.float32)])
    vertices = np.vstack([front_verts, back_verts])

    # 2. Distinct rainbow/spectral colors for each vertex
    palette_10 = np.array([
        [0.9, 0.9, 0.9],  # 0: J (bright white-gray hub)
        [0.8, 0.4, 0.0],  # 1: E (amber)
        [0.9, 0.9, 0.0],  # 2: F (yellow)
        [0.2, 0.9, 0.1],  # 3: G (green)
        [0.0, 0.9, 0.8],  # 4: I (cyan)
        [0.1, 0.3, 0.9],  # 5: H (blue)
        [0.5, 0.1, 0.9],  # 6: D (violet)
        [0.9, 0.1, 0.6],  # 7: C (magenta)
        [0.8, 0.0, 0.1],  # 8: A (crimson)
        [0.6, 0.0, 0.0]   # 9: B (dark red)
    ], dtype=np.float32)
    colors = np.vstack([palette_10, palette_10])

    # 3. Triangles (front provided; back and sides are the TODOs)
    # Front fan: 9 triangles radiating from vertex 0 (J) around the boundary
    front_tris = [
        [0, 1, 2], [0, 2, 3], [0, 3, 4], [0, 4, 5],
        [0, 5, 6], [0, 6, 7], [0, 7, 8], [0, 8, 9], [0, 9, 1]
    ]

    # TODO 1 (Geometry): Generate the nine back-face triangles.
    # The back vertices have indices 10 through 19.
    # Reverse the winding of each front triangle so its normal points outward.
    # Example: front [0, 1, 2] -> back [10, 12, 11].
    back_tris = []  # TODO: replace with your computed triangles

    # Side quads (2 triangles per boundary edge)
    side_tris = []
    boundary_cycle = [1, 2, 3, 4, 5, 6, 7, 8, 9, 1]
    for idx in range(len(boundary_cycle) - 1):
        i = boundary_cycle[idx]
        j = boundary_cycle[idx + 1]
        i_back = i + 10
        j_back = j + 10
        # TODO 2 (Geometry): Split this boundary quad into two triangles.
        # Use i, j, i_back, and j_back with outward-facing winding.
        # side_tris.append([...])
        # side_tris.append([...])
        pass

    indices = np.array(front_tris + back_tris + side_tris, dtype=np.uint32).flatten()
    return vertices, colors, indices

# ==============================================================================
# Camera & Linear Algebra Helpers
# ==============================================================================

class ArcballCamera:
    def __init__(self, distance=3.0):
        self.distance = distance
        self.rot_x = 20.0   # Pitch (degrees)
        self.rot_y = -35.0  # Yaw (degrees)
        self.last_x = 0.0
        self.last_y = 0.0
        self.is_dragging = False

    def on_mouse_press(self, x, y):
        self.is_dragging = True
        self.last_x = x
        self.last_y = y

    def on_mouse_release(self):
        self.is_dragging = False

    def on_mouse_drag(self, x, y):
        if not self.is_dragging:
            return
        dx = x - self.last_x
        dy = y - self.last_y
        self.last_x = x
        self.last_y = y

        # TODO 4 (Camera): Update yaw using dx and pitch using dy.
        # Suggested sensitivity: 0.4 degrees per pixel.
        # Clamp pitch to [-89, 89] degrees.
        pass

    def get_modelview(self):
        rad_x = np.radians(self.rot_x)
        rad_y = np.radians(self.rot_y)

        # Rotation around X (Pitch)
        Rx = np.array([
            [1.0, 0.0,            0.0,           0.0],
            [0.0, np.cos(rad_x), -np.sin(rad_x), 0.0],
            [0.0, np.sin(rad_x),  np.cos(rad_x), 0.0],
            [0.0, 0.0,            0.0,           1.0]
        ], dtype=np.float32)

        # Rotation around Y (Yaw)
        Ry = np.array([
            [ np.cos(rad_y), 0.0, np.sin(rad_y), 0.0],
            [ 0.0,           1.0, 0.0,           0.0],
            [-np.sin(rad_y), 0.0, np.cos(rad_y), 0.0],
            [ 0.0,           0.0, 0.0,           1.0]
        ], dtype=np.float32)

        # Translation back along Z
        T = np.identity(4, dtype=np.float32)
        T[2, 3] = -self.distance

        # TODO 5 (Camera): Compose translation, pitch, and yaw.
        # The expected order is translation @ pitch @ yaw.
        # Return identity initially so the starter remains runnable.
        return T  # TODO: replace with composed matrix

def make_perspective(fov_rad, aspect, near, far):
    tan_half_fov = np.tan(fov_rad / 2.0)
    res = np.zeros((4, 4), dtype=np.float32)
    res[0, 0] = 1.0 / (aspect * tan_half_fov)
    res[1, 1] = 1.0 / tan_half_fov
    res[2, 2] = -(far + near) / (far - near)
    res[2, 3] = -(2.0 * far * near) / (far - near)
    res[3, 2] = -1.0
    return res

# ==============================================================================
# Shader Compilation & Linkage
# ==============================================================================

def compile_shader(src, shader_type):
    s = glCreateShader(shader_type)
    glShaderSource(s, src)
    glCompileShader(s)
    if not glGetShaderiv(s, GL_COMPILE_STATUS):
        raise RuntimeError(glGetShaderInfoLog(s).decode())
    return s

def build_program():
    vs = compile_shader(VERTEX_SHADER_SRC, GL_VERTEX_SHADER)
    fs = compile_shader(FRAGMENT_SHADER_SRC, GL_FRAGMENT_SHADER)
    prog = glCreateProgram()
    glAttachShader(prog, vs)
    glAttachShader(prog, fs)
    glLinkProgram(prog)
    if not glGetProgramiv(prog, GL_LINK_STATUS):
        raise RuntimeError(glGetProgramInfoLog(prog).decode())
    glDeleteShader(vs)
    glDeleteShader(fs)
    return prog

# ==============================================================================
# Main Execution
# ==============================================================================

def main():
    if not glfw.init():
        sys.exit("Failed to initialize GLFW.")

    glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 3)
    glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
    glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)

    window = glfw.create_window(800, 700, "HW4 Starter: 3D Letter Shading and Camera", None, None)
    if not window:
        glfw.terminate()
        sys.exit("Failed to create GLFW window.")

    glfw.make_context_current(window)
    # TODO 3a (Visibility): Enable depth testing here.
    # Hint: use glEnable with the appropriate OpenGL capability.

    camera = ArcballCamera(distance=2.8)

    # GLFW Mouse Callbacks
    def mouse_button_cb(w, button, action, mods):
        if button == glfw.MOUSE_BUTTON_LEFT:
            if action == glfw.PRESS:
                x, y = glfw.get_cursor_pos(w)
                camera.on_mouse_press(x, y)
            elif action == glfw.RELEASE:
                camera.on_mouse_release()

    def cursor_pos_cb(w, xpos, ypos):
        camera.on_mouse_drag(xpos, ypos)

    glfw.set_mouse_button_callback(window, mouse_button_cb)
    glfw.set_cursor_pos_callback(window, cursor_pos_cb)

    # Provided: geometry upload and GPU buffer setup
    # Build geometry buffers
    verts, colors, indices = build_extruded_letter_b()

    vao = glGenVertexArrays(1)
    glBindVertexArray(vao)

    # Interleave attributes: [x, y, z, r, g, b]
    interleaved = np.empty((len(verts), 6), dtype=np.float32)
    interleaved[:, 0:3] = verts
    interleaved[:, 3:6] = colors

    vbo = glGenBuffers(1)
    glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, interleaved.nbytes, interleaved, GL_STATIC_DRAW)

    ebo = glGenBuffers(1)
    glBindBuffer(GL_ELEMENT_ARRAY_BUFFER, ebo)
    glBufferData(GL_ELEMENT_ARRAY_BUFFER, indices.nbytes, indices, GL_STATIC_DRAW)

    stride = 6 * interleaved.itemsize
    # Pos = location 0
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(0))
    glEnableVertexAttribArray(0)
    # Color = location 1
    glVertexAttribPointer(1, 3, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(3 * 4))
    glEnableVertexAttribArray(1)

    glBindVertexArray(0)

    prog = build_program()
    loc_mv = glGetUniformLocation(prog, "uModelView")
    loc_pj = glGetUniformLocation(prog, "uPersp")

    while not glfw.window_should_close(window):
        w, h = glfw.get_framebuffer_size(window)
        glViewport(0, 0, w, h)
        aspect = w / float(h if h > 0 else 1)

        glClearColor(0.12, 0.14, 0.18, 1.0)
        # TODO 3b (Visibility): Clear BOTH the color and depth buffers.
        # Initially only the color buffer is cleared.
        glClear(GL_COLOR_BUFFER_BIT)

        MV = camera.get_modelview()
        P = make_perspective(np.radians(40.0), aspect, 0.1, 50.0)

        glUseProgram(prog)
        glUniformMatrix4fv(loc_mv, 1, GL_FALSE, MV.T.astype(np.float32))
        glUniformMatrix4fv(loc_pj, 1, GL_FALSE, P.T.astype(np.float32))

        glBindVertexArray(vao)
        glDrawElements(GL_TRIANGLES, len(indices), GL_UNSIGNED_INT, None)
        glBindVertexArray(0)

        glfw.swap_buffers(window)
        glfw.poll_events()

    glDeleteVertexArrays(1, [vao])
    glDeleteBuffers(1, [vbo])
    glDeleteBuffers(1, [ebo])
    glDeleteProgram(prog)
    glfw.terminate()

if __name__ == "__main__":
    main()

: 